In [38]:
# 1. INSTALL DEPENDENCIES (Run this in a separate cell if using a Jupyter Notebook)
%pip install langchain_huggingface langchain_qdrant qdrant_client openai python-dotenv ollama sentence-transformers langchain-ollama -q


In [24]:
!ollama run llama3.1

>>> Send a message (/? for help)^C


In [35]:
!ollama pull all-minilm:22m

In [36]:
!ollama list

NAME               ID              SIZE      MODIFIED       
all-minilm:22m     1b226e2802db    45 MB     2 seconds ago     
llama3.1:latest    46e0c10c039e    4.9 GB    31 minutes ago    


In [ ]:
import os
from dotenv import load_dotenv
from langchain_qdrant import QdrantVectorStore
load_dotenv()  # Load environment variables from .env file
# from openai import OpenAI
from qdrant_client import QdrantClient
from ollama import chat
from langchain_ollama import OllamaEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Client for Google Gemini API initialization from the OpenAI class
# client = OpenAI(
#     api_key=os.environ["GOOGLE_API_KEY"],
#     base_url=

#     "https://generativelanguage.googleapis.com/v1beta/openai/",
# )

embedding_model = OllamaEmbeddings(
    model="all-minilm:22m",
)

# embedding_model = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-small-en-v1.5",
#     model_kwargs={"device": "cuda"},  # Change to 'cuda' if you have an Nvidia GPU
#     encode_kwargs={
#         "normalize_embeddings": True
#     },  # Essential for BGE cosine distance calculation
# )


qclient = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    check_compatibility=False,
)  # Qdrant local server URL

vector_db = QdrantVectorStore(
    embedding=embedding_model,
    client=qclient,  # Use the Qdrant client for connection
    collection_name="thesis_hf_cloud",  # Re-ingested collection name for 384 dimensions
)


while True:
    # Take the user input and query the vector store
    user_input = input("Ask Something: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        break
    # Relevant chunks from the vector db
    search_results = vector_db.similarity_search(
        user_input, k=3
    )  # k is the number of relevant chunks to retrieve

    context = "\n\n\n".join(
        [
            f"Page Content:{result.page_content}\nPage Number: {result.metadata.get('page_label', 'N/A')}\nFile Location: {result.metadata.get('source', 'N/A')}"
            for result in search_results
        ]
    )

    SYSTEM_PROMPT = f"""You are a helpful assistant who answers questions based on the available context
    retrieved from a PDF file along with page contents and page numbers.

    You only answer the user based on the following context and navigate the
    user to the open the right page number to know more.

    Context:
    {context}
    """
    print(f"\n\nyou:{user_input}\n")
    print("Agent thinking...\n")
    response = chat(
        model="llama3.1",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input},
        ],
        stream=True,
    )
    for chunk in response:
        print(chunk["message"]["content"], end="", flush=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]



you:hi

Agent thinking...

It seems you're looking at a thesis document with a table of contents. What would you like to know or where would you like to start? The table of contents shows the different sections and their corresponding page numbers.

you:what is the name of this book

Agent thinking...

Unfortunately, I don't see the title of the book in the provided context. However, I can suggest some alternatives to help you find the answer:

1. Please open page 1 of the PDF file, which is mentioned in the table of contents (SL NO: 1, TITLE: INTRODUCTION, PAGE NUMBER: 1).
2. If you open page 1, you should be able to see the title of the book.

If you need help with anything else, feel free to ask!

you:okay who wrote this

Agent thinking...

According to the context, the author of this thesis is:

Tantawy, Mohamed F.

You can find more information about the author on page 1 of the thesis, which is listed under the title "INTRODUCTION". Would you like to open page 1 to know more?

y